<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/MovieLens_TwinTowers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title COMPLETE Industrial two-tower retrieval (MovieLens-1M) + in-batch/exp/hard/rand negatives + correct CE + epoch schedules + per-epoch VALIDATION
# ✅ Simple false-negative control (FAST): precompute per-user SUSPECT set via genre-signature buckets, then sample 95% from SAFE + 5% from SUSPECT for HARD/RAND
!pip -q install pandas numpy torch tqdm scikit-learn

import os, random, zipfile, urllib.request
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# Repro + device
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ----------------------------
# Download MovieLens 1M
# ----------------------------
DATA_DIR = "/content/ml-1m"
if not os.path.exists(DATA_DIR):
    os.makedirs("/content", exist_ok=True)
    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    zip_path = "/content/ml-1m.zip"
    print("Downloading:", url)
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")
    print("Extracted to:", DATA_DIR)

users = pd.read_csv(f"{DATA_DIR}/users.dat", sep="::", header=None, engine="python",
                    names=["userId","gender","age","occupation","zip"])
movies = pd.read_csv(f"{DATA_DIR}/movies.dat", sep="::", header=None, engine="python",
                     names=["movieId","title","genres"], encoding="latin-1")
ratings = pd.read_csv(f"{DATA_DIR}/ratings.dat", sep="::", header=None, engine="python",
                      names=["userId","movieId","rating","timestamp"])

# ----------------------------
# Time-based split per user:
# last -> test, second last -> val, rest -> train
# ----------------------------
ratings = ratings.sort_values(["userId","timestamp"])
grp = ratings.groupby("userId", group_keys=False)

def split_last_two(df):
    if len(df) < 3:
        df = df.copy()
        df["split"] = "train"
        return df
    df = df.copy()
    df["split"] = "train"
    df.iloc[-1, df.columns.get_loc("split")] = "test"
    df.iloc[-2, df.columns.get_loc("split")] = "val"
    return df

ratings = grp.apply(split_last_two)

train_all = ratings[ratings["split"]=="train"][["userId","movieId","rating","timestamp"]]
val_all   = ratings[ratings["split"]=="val"][["userId","movieId","rating","timestamp"]]
test_all  = ratings[ratings["split"]=="test"][["userId","movieId","rating","timestamp"]]

print("all interactions:", len(ratings), "train:", len(train_all), "val:", len(val_all), "test:", len(test_all))

# ----------------------------
# Labels (implicit):
# Positive: rating>=4
# Exposed negative (TRAIN ONLY): ✅ start easy: use rating<=2 (treat rating==3 as "unknown/neutral")
# ----------------------------
train_pos = train_all[train_all["rating"] >= 4][["userId","movieId","timestamp"]]
train_exp_neg = train_all[train_all["rating"] <= 2][["userId","movieId","timestamp"]]

val_pos = val_all[val_all["rating"] >= 4][["userId","movieId","timestamp"]]
test_pos = test_all[test_all["rating"] >= 4][["userId","movieId","timestamp"]]

print("train_pos:", len(train_pos), "train_exposed_neg(<=2):", len(train_exp_neg))
print("val_pos:", len(val_pos), "test_pos:", len(test_pos))

# ----------------------------
# Build ID maps (full catalog)
# ----------------------------
all_user_ids = users["userId"].unique()
all_movie_ids = movies["movieId"].unique()

u2idx = {u:i for i,u in enumerate(all_user_ids)}
m2idx = {m:i for i,m in enumerate(all_movie_ids)}

users["u"] = users["userId"].map(u2idx)
movies["m"] = movies["movieId"].map(m2idx)

num_users = len(u2idx)
num_items = len(m2idx)
print("num_users:", num_users, "num_items:", num_items)

# ----------------------------
# Discrete features -> embeddings
# User: gender, age, occupation
# Item: genres (each genre -> emb -> aggregate)
# ----------------------------
gender_vocab = {g:i for i,g in enumerate(sorted(users["gender"].unique()))}
age_vocab    = {a:i for i,a in enumerate(sorted(users["age"].unique()))}
occ_vocab    = {o:i for i,o in enumerate(sorted(users["occupation"].unique()))}

users["gender_idx"] = users["gender"].map(gender_vocab).astype(int)
users["age_idx"]    = users["age"].map(age_vocab).astype(int)
users["occ_idx"]    = users["occupation"].map(occ_vocab).astype(int)

movies["genre_list"] = movies["genres"].str.split("|")
all_genres = sorted({g for gl in movies["genre_list"] for g in gl})
genre_vocab = {g:i for i,g in enumerate(all_genres)}
movies["genre_ids"] = movies["genre_list"].apply(lambda gl: [genre_vocab[g] for g in gl])

# Per-user feature tensors (indexed by user-idx)
u_gender = torch.zeros(num_users, dtype=torch.long)
u_age    = torch.zeros(num_users, dtype=torch.long)
u_occ    = torch.zeros(num_users, dtype=torch.long)
for row in users.itertuples(index=False):
    u_gender[row.u] = row.gender_idx
    u_age[row.u]    = row.age_idx
    u_occ[row.u]    = row.occ_idx

# Per-item genres list-of-ids (indexed by item-idx)
item_genres = [None]*num_items
for row in movies.itertuples(index=False):
    item_genres[row.m] = row.genre_ids

def pad_genres(item_indices, device):
    lists = [item_genres[i] for i in item_indices]
    L = max(len(x) for x in lists)
    pad = np.zeros((len(lists), L), dtype=np.int64)
    mask = np.zeros((len(lists), L), dtype=np.float32)
    for i, gl in enumerate(lists):
        pad[i, :len(gl)] = gl
        mask[i, :len(gl)] = 1.0
    return torch.tensor(pad, dtype=torch.long, device=device), torch.tensor(mask, dtype=torch.float32, device=device)

# ----------------------------
# TRAIN-only history (no leakage)
# ----------------------------
user_pos_train = {u:set() for u in range(num_users)}
user_expneg_train = {u:set() for u in range(num_users)}

for r in train_pos.itertuples(index=False):
    user_pos_train[u2idx[r.userId]].add(m2idx[r.movieId])

for r in train_exp_neg.itertuples(index=False):
    user_expneg_train[u2idx[r.userId]].add(m2idx[r.movieId])

# ----------------------------
# ✅ FAST false-negative control: precompute per-user SUSPECT set using genre-signature buckets
# - Conservative: exact same genre signature as any positive item of the user
# - Then: sample 95% from SAFE (not suspect) + 5% from SUSPECT (difficulty spice)
# ----------------------------
genre_sig = [frozenset(gs) for gs in item_genres]
sig2items = {}
for m, sig in enumerate(genre_sig):
    sig2items.setdefault(sig, []).append(m)

user_suspect = {u:set() for u in range(num_users)}
for u in range(num_users):
    for p in user_pos_train[u]:
        user_suspect[u].update(sig2items.get(genre_sig[p], []))
    user_suspect[u].difference_update(user_pos_train[u])  # don't include positives themselves

def sample_mixture_neg(u, n, p_sus=0.05, max_tries_factor=80):
    """
    Sample n negatives for user u:
      - (1-p_sus) from SAFE (not in user_suspect[u])
      - p_sus from SUSPECT
    Always excludes positives.
    Falls back gracefully if SAFE is scarce.
    """
    sus = user_suspect[u]
    out = []

    n_sus = int(round(n * p_sus))
    n_safe = n - n_sus

    # SAFE sampling
    tries = 0
    max_tries = max(100, n_safe * max_tries_factor)
    while len(out) < n_safe and tries < max_tries:
        m = random.randrange(num_items)
        if (m not in user_pos_train[u]) and (m not in sus):
            out.append(m)
        tries += 1

    # fallback: fill safe part with any non-positive
    while len(out) < n_safe:
        m = random.randrange(num_items)
        if m not in user_pos_train[u]:
            out.append(m)

    # SUSPECT sampling
    if n_sus > 0:
        sus_list = list(sus)
        if len(sus_list) > 0:
            take = min(n_sus, len(sus_list))
            out += random.sample(sus_list, take)
            # if still short, random fill
            while len(out) < n:
                m = random.randrange(num_items)
                if m not in user_pos_train[u]:
                    out.append(m)
        else:
            # no suspect pool, random fill
            while len(out) < n:
                m = random.randrange(num_items)
                if m not in user_pos_train[u]:
                    out.append(m)

    return out[:n]

def pick_hard_with_mixture(u, hard_list, n, p_sus=0.05):
    """
    Hard negatives are already high-score candidates.
    Prefer SAFE hard negatives, allow a small fraction from SUSPECT,
    then fall back to mixture random.
    """
    sus = user_suspect[u]

    # remove positives, keep only unique candidates
    seen = set()
    hard_list = [m for m in hard_list if (m not in user_pos_train[u]) and (m not in seen) and (not seen.add(m))]
    if len(hard_list) == 0:
        return sample_mixture_neg(u, n, p_sus=p_sus)

    safe_h = [m for m in hard_list if m not in sus]
    sus_h  = [m for m in hard_list if m in sus]

    n_sus = int(round(n * p_sus))
    n_safe = n - n_sus

    out = []
    if len(safe_h) > 0:
        out += random.sample(safe_h, k=min(n_safe, len(safe_h)))

    # if safe hard不足，用sus_h补到n_safe（让hard仍然hard）
    if len(out) < n_safe and len(sus_h) > 0:
        need = n_safe - len(out)
        cand = [m for m in sus_h if m not in out]
        if cand:
            out += random.sample(cand, k=min(need, len(cand)))

    # add suspect quota (optional)
    if n_sus > 0 and len(sus_h) > 0:
        remaining = n - len(out)
        cand = [m for m in sus_h if m not in out]
        if cand and remaining > 0:
            out += random.sample(cand, k=min(remaining, n_sus, len(cand)))

    # final fill with mixture random if still short
    if len(out) < n:
        out += sample_mixture_neg(u, n - len(out), p_sus=p_sus)

    return out[:n]

# ----------------------------
# Eval ground-truth (held-out positive only)
# Users whose val/test heldout is not positive are skipped.
# ----------------------------
val_gt  = {u:None for u in range(num_users)}
test_gt = {u:None for u in range(num_users)}

for r in val_pos.itertuples(index=False):
    val_gt[u2idx[r.userId]] = m2idx[r.movieId]
for r in test_pos.itertuples(index=False):
    test_gt[u2idx[r.userId]] = m2idx[r.movieId]

val_users  = [u for u,g in val_gt.items() if g is not None]
test_users = [u for u,g in test_gt.items() if g is not None]
print("val_users (has positive):", len(val_users), "test_users (has positive):", len(test_users))

# ----------------------------
# Per-user TRAIN-positive pool + epoch plan covering ALL users (with repeats)
# ----------------------------
user_pos_list = {u: [] for u in range(num_users)}
for u, pos_set in user_pos_train.items():
    if len(pos_set) > 0:
        user_pos_list[u] = list(pos_set)

train_users = [u for u,lst in user_pos_list.items() if len(lst) > 0]
print("train_users_with_pos:", len(train_users))

REPEATS_PER_USER = 20

def make_epoch_user_plan():
    plan = train_users[:] * REPEATS_PER_USER
    random.shuffle(plan)
    return plan

def iter_user_batches(plan, batch_size):
    for i in range(0, len(plan), batch_size):
        yield plan[i:i+batch_size]

def make_batch_users_unique(us, batch_size):
    seen = set()
    uniq = []
    for u in us:
        if u not in seen:
            seen.add(u)
            uniq.append(u)
        if len(uniq) == batch_size:
            break
    while len(uniq) < batch_size:
        u = random.choice(train_users)
        if u not in seen:
            seen.add(u)
            uniq.append(u)
    return uniq

def sample_pos_for_users(us):
    return [random.choice(user_pos_list[u]) for u in us]

# ----------------------------
# Residual MLP blocks
# ----------------------------
class ResBlock(nn.Module):
    def __init__(self, d, hidden=None, dropout=0.1):
        super().__init__()
        hidden = hidden or (d * 2)
        self.fc1 = nn.Linear(d, hidden)
        self.fc2 = nn.Linear(hidden, d)
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(d)
    def forward(self, x):
        h = F.gelu(self.fc1(x))
        h = self.dropout(self.fc2(h))
        return self.ln(x + h)

class ResMLP(nn.Module):
    def __init__(self, d_in, d_out, d=256, depth=4, dropout=0.1):
        super().__init__()
        self.proj_in = nn.Linear(d_in, d)
        self.blocks = nn.Sequential(*[ResBlock(d, dropout=dropout) for _ in range(depth)])
        self.proj_out = nn.Linear(d, d_out)
    def forward(self, x):
        x = self.proj_in(x)
        x = self.blocks(x)
        x = self.proj_out(x)
        return x

# ----------------------------
# Genre pooling
# ----------------------------
class GenreAttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.q = nn.Linear(d, 1, bias=False)
    def forward(self, ge, mask):
        logits = self.q(ge).squeeze(-1)     # [B,L]
        logits = logits.masked_fill(mask <= 0, -1e9)
        w = torch.softmax(logits, dim=1)    # [B,L]
        pooled = torch.einsum("bl,bld->bd", w, ge)
        return pooled

# ----------------------------
# Two-tower
# ----------------------------
class TwoTower(nn.Module):
    def __init__(self,
                 num_users, num_items,
                 num_gender, num_age, num_occ,
                 num_genres,
                 d_id=64, d_feat=16,
                 d_tower=256, depth=4, dropout=0.10,
                 d_out=128,
                 genre_pool="mean"):  # "mean" or "attn"
        super().__init__()
        self.user_id_emb = nn.Embedding(num_users, d_id)
        self.item_id_emb = nn.Embedding(num_items, d_id)

        self.gender_emb = nn.Embedding(num_gender, d_feat)
        self.age_emb    = nn.Embedding(num_age, d_feat)
        self.occ_emb    = nn.Embedding(num_occ, d_feat)

        self.genre_emb  = nn.Embedding(num_genres, d_feat)

        self.genre_pool = genre_pool
        self.genre_pooler = GenreAttnPool(d_feat) if genre_pool == "attn" else None

        user_in = d_id + 3*d_feat
        item_in = d_id + d_feat

        self.user_tower = ResMLP(user_in, d_out, d=d_tower, depth=depth, dropout=dropout)
        self.item_tower = ResMLP(item_in, d_out, d=d_tower, depth=depth, dropout=dropout)

    def encode_user(self, u_idx):
        uid = self.user_id_emb(u_idx)
        g = self.gender_emb(u_gender[u_idx].to(u_idx.device))
        a = self.age_emb(u_age[u_idx].to(u_idx.device))
        o = self.occ_emb(u_occ[u_idx].to(u_idx.device))
        x = torch.cat([uid, g, a, o], dim=-1)
        z = self.user_tower(x)
        return F.normalize(z, dim=-1)

    def encode_item(self, m_idx, genre_pad, genre_mask):
        iid = self.item_id_emb(m_idx)
        ge  = self.genre_emb(genre_pad)  # [B,L,d_feat]
        mask = genre_mask                # [B,L]

        if self.genre_pool == "attn":
            pooled = self.genre_pooler(ge, mask)
        else:
            mask_f = mask.unsqueeze(-1)
            denom = mask_f.sum(dim=1).clamp_min(1.0)
            pooled = (ge * mask_f).sum(dim=1) / denom

        x = torch.cat([iid, pooled], dim=-1)
        z = self.item_tower(x)
        return F.normalize(z, dim=-1)

model = TwoTower(
    num_users=num_users,
    num_items=num_items,
    num_gender=len(gender_vocab),
    num_age=len(age_vocab),
    num_occ=len(occ_vocab),
    num_genres=len(genre_vocab),
    d_id=64, d_feat=16,
    d_tower=256, depth=4, dropout=0.10,
    d_out=128,
    genre_pool="mean"  # or "attn"
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ----------------------------
# Negative samplers (EXPOSED remains from labels)
# ----------------------------
def sample_random_neg(u, n):
    negs = []
    while len(negs) < n:
        m = random.randrange(num_items)
        if m not in user_pos_train[u]:
            negs.append(m)
    return negs

def sample_exposed_neg(u, n):
    pool = list(user_expneg_train[u])
    if len(pool) == 0:
        return sample_random_neg(u, n)
    if len(pool) >= n:
        return random.sample(pool, n)
    res = pool[:]
    res += sample_random_neg(u, n - len(res))
    return res

# ----------------------------
# Hard mining
# ----------------------------
@torch.no_grad()
def mine_hard_negs_for_user(u, pool_size=1500, hard_k=200):
    cand = set()
    exp_pool = list(user_expneg_train[u])
    if len(exp_pool) > 0:
        take = min(len(exp_pool), pool_size // 2)
        for m in random.sample(exp_pool, take):
            if m not in user_pos_train[u]:
                cand.add(m)
    while len(cand) < pool_size:
        m = random.randrange(num_items)
        if m not in user_pos_train[u]:
            cand.add(m)
    cand = list(cand)

    model.eval()
    u_t = torch.tensor([u], device=device, dtype=torch.long)
    uz = model.encode_user(u_t)

    m_t = torch.tensor(cand, device=device, dtype=torch.long)
    gpad, gmask = pad_genres(cand, device=device)
    mz = model.encode_item(m_t, gpad, gmask)

    scores = (uz @ mz.T).squeeze(0)
    top = torch.topk(scores, k=min(hard_k, len(cand))).indices.cpu().numpy()
    return [cand[i] for i in top]

# ----------------------------
# Eval: Recall@K
# ----------------------------
@torch.no_grad()
def compute_recall_at_k(user_list, gt_map, K_list=(5,10,20,50)):
    model.eval()
    all_items = torch.arange(num_items, device=device, dtype=torch.long)
    gpad_all, gmask_all = pad_genres(list(range(num_items)), device=device)
    item_z = model.encode_item(all_items, gpad_all, gmask_all)

    recalls = {K:[] for K in K_list}
    for u in tqdm(user_list, desc="eval"):
        gt = gt_map[u]
        u_t = torch.tensor([u], device=device, dtype=torch.long)
        uz = model.encode_user(u_t)
        scores = (uz @ item_z.T).squeeze(0)

        if len(user_pos_train[u]) > 0:
            scores[list(user_pos_train[u])] = -1e9

        top_items = torch.topk(scores, k=max(K_list)).indices.cpu().numpy().tolist()
        for K in K_list:
            recalls[K].append(1.0 if gt in top_items[:K] else 0.0)

    return {K: float(np.mean(v)) for K,v in recalls.items()}

# ----------------------------
# Explicit-neg CE helper: ONLY (u,pos) vs (u,negs-from-one-source)
# ----------------------------
def ce_one_pos_many_negs(pos_score, neg_scores):
    logits = torch.cat([pos_score, neg_scores], dim=1)  # [B, 1+N]
    labels0 = torch.zeros(logits.size(0), device=logits.device, dtype=torch.long)
    return F.cross_entropy(logits, labels0)

# ----------------------------
# Schedules: early in-batch dominant, later exp/hard/rnd dominant
# and temperature HIGH -> LOW (soft -> sharp)
# ----------------------------
EPOCHS = 20
BATCH = 2048

HARD_NEG = 20
EXP_NEG  = 20
RAND_NEG = 20

HARD_REFRESH_STEPS = 50
HARD_WARMUP_STEPS = 1500     # prevents very early noisy hard mining
HARD_RAMP_STEPS   = 1000     # ramps hard contribution smoothly after warmup

# Weight targets:
# early: inb=1.0, others=0.0
# final: inb=0.60, exp=0.20, hard=0.15, rnd=0.05
FINAL_W = dict(inb=0.40, exp=0.30, hard=0.25, rnd=0.05)
FREEZE_EPOCHS = 5

def get_epoch_weights(ep, EPOCHS, freeze_epochs=FREEZE_EPOCHS):
    if ep <= freeze_epochs:
        return 1.0, 0.0, 0.0, 0.0  # inb, exp, hard, rnd
    denom = max(1, EPOCHS - freeze_epochs)
    t = (ep - freeze_epochs) / float(denom)
    t = min(max(t, 0.0), 1.0)
    w_inb  = (1.0 - t) * 1.0 + t * FINAL_W["inb"]
    w_exp  = t * FINAL_W["exp"]
    w_hard = t * FINAL_W["hard"]
    w_rnd  = t * FINAL_W["rnd"]
    return w_inb, w_exp, w_hard, w_rnd

# Temperature schedule: HIGH -> LOW
TEMP_START = 0.15   # early (soft)
TEMP_END   = 0.07   # late  (sharp)
TEMP_FREEZE_EPOCHS = 0

def get_epoch_temperature(ep, EPOCHS, freeze_epochs=TEMP_FREEZE_EPOCHS):
    if ep <= freeze_epochs:
        return TEMP_START
    denom = max(1, EPOCHS - freeze_epochs)
    t = (ep - freeze_epochs) / float(denom)
    t = min(max(t, 0.0), 1.0)
    return TEMP_START + t * (TEMP_END - TEMP_START)  # decreases since END < START

hard_cache = {u: [] for u in range(num_users)}

# ----------------------------
# Mixture ratios (SAFE vs SUSPECT)
# - Use small SUSPECT fraction to keep difficulty but reduce false negatives.
# ----------------------------
P_SUS_HARD = 0.05
P_SUS_RAND = 0.05

# ----------------------------
# Train (with per-epoch validation)
# ----------------------------
step = 0
for ep in range(1, EPOCHS+1):
    model.train()

    w_inb, w_exp, w_hard_base, w_rnd = get_epoch_weights(ep, EPOCHS, freeze_epochs=FREEZE_EPOCHS)

    temp_ep = get_epoch_temperature(ep, EPOCHS, freeze_epochs=TEMP_FREEZE_EPOCHS)
    inv_t = 1.0 / temp_ep

    epoch_plan = make_epoch_user_plan()
    batch_stream = list(iter_user_batches(epoch_plan, BATCH))
    pbar = tqdm(batch_stream, desc=f"train ep{ep} (T={temp_ep:.3f} | w_inb={w_inb:.2f} w_exp={w_exp:.2f} w_hard={w_hard_base:.2f} w_rnd={w_rnd:.2f})")

    for u_raw in pbar:
        step += 1

        u = make_batch_users_unique(u_raw, len(u_raw))
        pos = sample_pos_for_users(u)
        B = len(u)

        enable_hard = (step > HARD_WARMUP_STEPS)

        # Refresh hard cache periodically (after warmup)
        if enable_hard and (step % HARD_REFRESH_STEPS == 0):
            sample_users = u if len(u) <= 200 else random.sample(u, 200)
            for uu in sample_users:
                hard_cache[uu] = mine_hard_negs_for_user(uu, pool_size=1500, hard_k=200)

        # Sample negatives in order: [hard][exposed][random]
        neg = []
        for uu in u:
            # HARD (prefer SAFE-hard, allow small SUSPECT fraction, then fallback)
            if enable_hard:
                hard_list = hard_cache.get(uu, [])
                hard_pick = pick_hard_with_mixture(uu, hard_list, HARD_NEG, p_sus=P_SUS_HARD) if hard_list else sample_mixture_neg(uu, HARD_NEG, p_sus=P_SUS_HARD)
            else:
                hard_pick = sample_mixture_neg(uu, HARD_NEG, p_sus=P_SUS_HARD)

            # EXP (keep as-is; already from labels)
            exp_pick = sample_exposed_neg(uu, EXP_NEG)

            # RAND (95% SAFE + 5% SUSPECT)
            rnd_pick = sample_mixture_neg(uu, RAND_NEG, p_sus=P_SUS_RAND)

            neg.append(hard_pick + exp_pick + rnd_pick)

        # Tensorize
        u_t   = torch.tensor(u,   device=device, dtype=torch.long)
        pos_t = torch.tensor(pos, device=device, dtype=torch.long)
        neg_t = torch.tensor(neg, device=device, dtype=torch.long)  # [B, N]

        # Encode
        uz = model.encode_user(u_t)  # [B,d]

        gpad_pos, gmask_pos = pad_genres(pos, device=device)
        pz = model.encode_item(pos_t, gpad_pos, gmask_pos)  # [B,d]

        neg_flat = neg_t.reshape(-1).tolist()
        gpad_neg, gmask_neg = pad_genres(neg_flat, device=device)
        nz = model.encode_item(neg_t.reshape(-1), gpad_neg, gmask_neg).reshape(B, -1, uz.size(-1))  # [B,N,d]

        # Cosine scores with temperature scaling
        pos_score  = (uz * pz).sum(dim=-1, keepdim=True) * inv_t      # [B,1]
        neg_scores = torch.einsum("bd,bnd->bn", uz, nz) * inv_t       # [B,N]

        # (1) In-batch CE (positives in batch)
        logits_inbatch = (uz @ pz.T) * inv_t                          # [B,B]
        labels_inbatch = torch.arange(B, device=device, dtype=torch.long)
        loss_inbatch = F.cross_entropy(logits_inbatch, labels_inbatch)

        # (2) Per-source explicit neg CE (pos vs that source negs only)
        hard_scores = neg_scores[:, :HARD_NEG]
        exp_scores  = neg_scores[:, HARD_NEG:HARD_NEG + EXP_NEG]
        rnd_scores  = neg_scores[:, HARD_NEG + EXP_NEG:HARD_NEG + EXP_NEG + RAND_NEG]

        loss_hard = ce_one_pos_many_negs(pos_score, hard_scores)
        loss_exp  = ce_one_pos_many_negs(pos_score, exp_scores)
        loss_rnd  = ce_one_pos_many_negs(pos_score, rnd_scores)

        # Hard gate (step-level) on top of epoch schedule
        if step <= HARD_WARMUP_STEPS:
            hard_gate = 0.0
        elif step <= HARD_WARMUP_STEPS + HARD_RAMP_STEPS:
            hard_gate = (step - HARD_WARMUP_STEPS) / float(HARD_RAMP_STEPS)
        else:
            hard_gate = 1.0
        w_hard = w_hard_base * hard_gate

        # Stable normalization
        wsum = (w_inb + w_exp + w_hard + w_rnd)
        if wsum <= 0:
            wsum = 1.0

        loss = (w_inb * loss_inbatch + w_hard * loss_hard + w_exp * loss_exp + w_rnd * loss_rnd) / wsum

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()

        if step % 50 == 0:
            pbar.set_postfix(
                loss=float(loss.item()),
                inb=float(loss_inbatch.item()),
                hard=float(loss_hard.item()),
                exp=float(loss_exp.item()),
                rnd=float(loss_rnd.item()),
                T=float(temp_ep),
                hard_on=int(enable_hard)
            )

    # ✅ Validation at end of each epoch
    val_recall = compute_recall_at_k(val_users, val_gt, K_list=(5,10,20,50))
    print(f"\n[Epoch {ep}] Val Recall:", val_recall)

# Final test
test_recall = compute_recall_at_k(test_users, test_gt, K_list=(5,10,20,50))
print("\nFINAL Test Recall:", test_recall)


device: cpu
Downloading: https://files.grouplens.org/datasets/movielens/ml-1m.zip
Extracted to: /content/ml-1m


/tmp/ipython-input-3419569489.py:65: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ratings = grp.apply(split_last_two)


all interactions: 1000209 train: 988129 val: 6040 test: 6040
train_pos: 568232 train_exposed_neg(<=2): 161606
val_pos: 3486 test_pos: 3563
num_users: 6040 num_items: 3883
val_users (has positive): 3486 test_users (has positive): 3563
train_users_with_pos: 6038


train ep1 (T=0.146 | w_inb=1.00 w_exp=0.00 w_hard=0.00 w_rnd=0.00): 100%|██████████| 59/59 [08:40<00:00,  8.82s/it, T=0.146, exp=3.03, hard=3.01, hard_on=0, inb=7.6, loss=7.6, rnd=3.01]
eval: 100%|██████████| 3486/3486 [00:04<00:00, 798.47it/s]



[Epoch 1] Val Recall: {5: 0.002008032128514056, 10: 0.0034423407917383822, 20: 0.00717154331612163, 50: 0.02094090648307516}


...


[Epoch 19] Val Recall: {5: 0.025817555938037865, 10: 0.047905909351692484, 20: 0.08261617900172118, 50: 0.17096959265633965}


train ep20 (T=0.070 | w_inb=0.40 w_exp=0.30 w_hard=0.25 w_rnd=0.05): 100%|██████████| 59/59 [08:45<00:00,  8.91s/it, T=0.07, exp=1.39, hard=1.12, hard_on=0, inb=6.62, loss=4.16, rnd=1.12]
eval: 100%|██████████| 3486/3486 [00:04<00:00, 860.33it/s]



[Epoch 20] Val Recall: {5: 0.025817555938037865, 10: 0.047619047619047616, 20: 0.08376362593230063, 50: 0.1706827309236948}


eval: 100%|██████████| 3563/3563 [00:04<00:00, 878.29it/s]


FINAL Test Recall: {5: 0.021049677238282348, 10: 0.04378332865562728, 20: 0.08335672186359809, 50: 0.17485265225933203}
